# HomeReach Melbourne — 07: Scheduled service indicators

**Part A:** reproduce the active-trip lookup for **13 October 2026** (Australia/Melbourne service-day time).
**Part B:** combine those trips with complete stop sequences and the reviewed stop-to-LGA assignments.

Use the existing **homereach** kernel. Run all cells from the beginning; this notebook is self-contained and does not rely on another notebook's memory. Preserve personal notes before replacing your earlier Part A file.

**Scope:** feeds 2, 3 and 4, the same 31 reviewed LGAs, and this exact audited archive.
We count **regular-pickup, non-terminal stop-time events**. These are a scheduled-supply proxy, not passengers, vehicles, route headways, door-to-door accessibility, punctuality, or recommendations.

**Windows:** (1) the complete selected GTFS service day, retaining times at/after 24:00; (2) 07:00 inclusive–09:00 exclusive of that service day. We do not wrap extended times modulo 24 or add trips belonging to adjacent service dates. Thus the first total is a *service-day* total, not strictly a midnight-to-midnight calendar-day total.

The four original Part A code cells are retained. Their intermediate `LGA service indicators calculated: False` and `Files exported: 0` messages apply to Part A only. The final Part B checkpoint describes the completed analysis.

**Writes:** only this notebook's derived tables/manifest under `data/interim/gtfs_service/` and its generated `docs/gtfs_service_indicators.md`. All input files, earlier notebooks, the registry and Streamlit app are unchanged. No package installation, download or Git action occurs.

**Temporal caution:** rents describe 2025Q3; this schedule describes an illustrative 2026-10-13 service date. Do not use the newer schedule as historical evidence in rental backtests.


## 1. Locate and verify the inputs

This cell confirms the source fingerprint and checks that the stop-assignment CSV still matches notebook 06's manifest. The report's timestamp can vary between machines; the CSV is checked against **your own local report**, not against a precomputed copy of its bytes from another machine.

Required paths:

```text
 data/raw/transport/gtfs.zip
 data/interim/gtfs_spatial/stop_lga_matches.csv
 data/interim/gtfs_spatial/spatial_join_summary.json
```

The fingerprints saved below are checked again after the calculations. A missing file or mismatched fingerprint is a reason to investigate—not to edit the expected fingerprint or create substitute data. No installation is performed.

In [1]:
print("Starting notebook 07 — active-trip lookup...", flush=True)

from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
from datetime import date, datetime
from typing import Any
import hashlib
import json
import sys

import pandas as pd
from IPython.display import display


def require(condition: bool, message: str) -> None:
    """Keep validation active even when Python assertions are disabled."""
    if not condition:
        raise ValueError(message)


def sha256_file(path: str | Path) -> str:
    """Hash a file incrementally without changing it."""
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook inside the existing HomeReach project.")


def read_json_object(path: Path) -> dict[str, Any]:
    try:
        content = json.loads(path.read_text(encoding="utf-8-sig"))
    except (OSError, json.JSONDecodeError) as error:
        raise ValueError(f"Cannot read the existing report: {path}") from error
    require(isinstance(content, dict), f"Expected a JSON object in {path.name}.")
    return content


ROOT = find_project_root()
GTFS_PATH = ROOT / "data/raw/transport/gtfs.zip"
STOP_PATH = ROOT / "data/interim/gtfs_spatial/stop_lga_matches.csv"
SPATIAL_REPORT_PATH = ROOT / "data/interim/gtfs_spatial/spatial_join_summary.json"
SERVICE_DATE = date(2026, 10, 13)
FEED_IDS = ("2", "3", "4")
EXPECTED_GTFS_SHA256 = "c88af3abcaf0872e45196dcc1d5af9bc20a82c54a00a9dfef46c809ec2e34814"

input_paths = [GTFS_PATH, STOP_PATH, SPATIAL_REPORT_PATH]
for path in input_paths:
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path.relative_to(ROOT)}. Complete notebook 06 with the actual sources first."
        )

print("Verifying the GTFS archive and previous stop-assignment output...", flush=True)
input_hashes = {path: sha256_file(path) for path in input_paths}
require(
    input_hashes[GTFS_PATH] == EXPECTED_GTFS_SHA256,
    "GTFS snapshot differs from the audit. Do not bypass this check or edit the hash.",
)
spatial_report = read_json_object(SPATIAL_REPORT_PATH)
require(
    spatial_report.get("source", {}).get("gtfs_sha256") == EXPECTED_GTFS_SHA256,
    "The spatial-join report belongs to a different GTFS snapshot.",
)
require(
    spatial_report.get("output_sha256", {}).get(STOP_PATH.name) == input_hashes[STOP_PATH],
    "The stop-assignment CSV differs from its spatial-join report. Stop and review the files.",
)
require(
    spatial_report.get("source", {}).get("selected_feeds") == list(FEED_IDS),
    "The spatial-join report covers different operational feeds.",
)
scope = spatial_report.get("scope", {})
require(
    scope.get("proposed_later_service_date") == SERVICE_DATE.isoformat()
    and scope.get("agency_timezone") == "Australia/Melbourne"
    and scope.get("service_date_filter_applied") is False,
    "The previous location-inventory scope/date is different from this reviewed stage.",
)
quality = spatial_report.get("quality", {})
require(
    quality.get("selected_stop_platform_records") == 25031
    and quality.get("duplicate_composite_keys") == 0
    and quality.get("csv_readback_matches") is True
    and quality.get("input_files_unchanged") is True,
    "The earlier stop-spatial-join checkpoint is incomplete or different.",
)

print("INPUT CHECKPOINT PASSED")
print("Python:", sys.executable)
print("pandas:", pd.__version__)
print("Project root:", ROOT)
print("Service date:", SERVICE_DATE)
print("GTFS snapshot and stop-assignment CSV verified.")


Starting notebook 07 — active-trip lookup...
Verifying the GTFS archive and previous stop-assignment output...
INPUT CHECKPOINT PASSED
Python: c:\Users\PhamDang\anaconda3\envs\homereach\python.exe
pandas: 3.0.3
Project root: D:\My project\homereach_melbourne
Service date: 2026-10-13
GTFS snapshot and stop-assignment CSV verified.


## 2. Load the smaller timetable tables

For each feed we read `agency.txt`, `routes.txt`, `trips.txt`, `calendar.txt` and `calendar_dates.txt`. We do not extract the archive onto disk and do not load its large stop-time table. IDs and date labels remain strings so that leading zeroes or other source formatting are not lost.

This reader is intentionally specific to the audited ZIP structure. All five files are present in the selected feeds. Other GTFS datasets may use a different valid calendar arrangement; a new archive needs a new audit rather than silently relaxing these checks.

A `frequencies.txt` table would require separate handling of frequency-based trips. It is absent from this audited snapshot, and this reader checks that it remains absent.

In [2]:
TABLE_NAMES = (
    "agency.txt", "routes.txt", "trips.txt", "calendar.txt", "calendar_dates.txt",
)
DAYS = (
    "monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday",
)
REQUIRED_COLUMNS = {
    "agency.txt": {"agency_id", "agency_timezone"},
    "routes.txt": {"route_id", "agency_id", "route_short_name", "route_long_name", "route_type"},
    "trips.txt": {"trip_id", "route_id", "service_id"},
    "calendar.txt": {"service_id", "start_date", "end_date", *DAYS},
    "calendar_dates.txt": {"service_id", "date", "exception_type"},
}


def read_feed_tables(feed_id: str) -> dict[str, pd.DataFrame]:
    """Read one audited feed's smaller tables directly from the nested ZIP."""
    require(feed_id in FEED_IDS, f"Feed {feed_id} is outside this notebook's reviewed scope.")
    member = f"{feed_id}/google_transit.zip"
    with ZipFile(GTFS_PATH) as outer:
        require(outer.namelist().count(member) == 1, f"Expected exactly one member: {member}")
        inner_bytes = BytesIO(outer.read(member))

    tables = {}
    with inner_bytes:
        with ZipFile(inner_bytes) as inner:
            names = inner.namelist()
            require(len(names) == len(set(names)), f"Duplicate ZIP members in feed {feed_id}.")
            require("frequencies.txt" not in names, "Frequency-based trips require a separate reviewed method.")
            for filename in TABLE_NAMES:
                require(names.count(filename) == 1, f"Expected exactly one {filename} in feed {feed_id}.")
                with inner.open(filename) as handle:
                    frame = pd.read_csv(
                        handle, dtype="string", keep_default_na=False,
                        encoding="utf-8-sig", low_memory=False,
                    )
                missing = REQUIRED_COLUMNS[filename].difference(frame.columns)
                require(not missing, f"{feed_id}/{filename}: missing columns {sorted(missing)}")
                tables[filename] = frame
    return tables


feed_tables = {}
inventory_rows = []
for feed_id in FEED_IDS:
    print(f"Reading feed {feed_id}...", flush=True)
    feed_tables[feed_id] = read_feed_tables(feed_id)
    require(
        set(feed_tables[feed_id]["agency.txt"]["agency_timezone"]) == {"Australia/Melbourne"},
        f"Unexpected agency timezone in feed {feed_id}.",
    )
    inventory_rows.append({
        "feed_id": feed_id,
        **{name.removesuffix(".txt") + "_rows": len(feed_tables[feed_id][name]) for name in TABLE_NAMES},
    })

display(pd.DataFrame(inventory_rows))
print("Smaller timetable tables loaded.")
print("stop_times.txt has not been scanned.")


Reading feed 2...
Reading feed 3...
Reading feed 4...


,feed_id,agency_rows,routes_rows,trips_rows,calendar_rows,calendar_dates_rows
0,2,1,36,40556,26,40
1,3,1,24,55599,12,46
2,4,1,762,215328,5823,88898


Smaller timetable tables loaded.
stop_times.txt has not been scanned.


## 3. Select active service IDs and test the calendar rule

The rule for one feed and one date is:

```text
regular services for that weekday and date range
    UNION explicitly added services for that date
    MINUS explicitly removed services for that date
    = active service IDs
```

GTFS exception type **1** adds service and type **2** removes it. We apply the rules separately inside each feed because calendar IDs can repeat between feeds.

The checks validate real `YYYYMMDD` dates before comparing the strings. This avoids relying on pandas' underlying timestamp resolution and avoids confusing microseconds with nanoseconds. It also detects invalid dates and reversed calendar intervals.

The fictional self-check removes `regular` and adds `special` on 13 October. It confirms that the exception wins over the ordinary weekly pattern. The next day uses `regular`, and a day outside the calendar uses neither. These checks are notebook checks, not new pytest tests.

In [3]:
def require_key(frame: pd.DataFrame, columns: list[str], label: str) -> None:
    """Require complete, unpadded identifiers and a unique primary key."""
    require(set(columns).issubset(frame.columns), f"{label}: missing key columns.")
    for column in columns:
        values = frame[column].astype("string")
        require(
            values.notna().all() and values.ne("").all()
            and values.eq(values.str.strip()).all(),
            f"{label}: blank or padded identifier in {column}.",
        )
    require(not frame.duplicated(columns).any(), f"{label}: duplicate key {columns}.")


def validate_gtfs_dates(values: pd.Series, label: str) -> None:
    text = values.astype("string")
    require(
        text.notna().all() and text.str.fullmatch(r"[0-9]{8}").all(),
        f"{label}: expected complete YYYYMMDD strings.",
    )
    for value in text.unique():
        try:
            datetime.strptime(str(value), "%Y%m%d")
        except ValueError as error:
            raise ValueError(f"{label}: invalid date {value!r}.") from error


def active_service_ids(
    calendar: pd.DataFrame | None,
    exceptions: pd.DataFrame | None,
    service_date: date,
) -> set[str]:
    """Select IDs within one feed, including exception-only service."""
    require(type(service_date) is date, "Use a date object for the local GTFS service day.")
    date_key = service_date.strftime("%Y%m%d")
    weekday = DAYS[service_date.weekday()]
    base, additions, removals = set(), set(), set()

    if calendar is not None and not calendar.empty:
        require(REQUIRED_COLUMNS["calendar.txt"].issubset(calendar.columns), "Missing calendar columns.")
        require_key(calendar, ["service_id"], "calendar")
        validate_gtfs_dates(calendar["start_date"], "calendar start")
        validate_gtfs_dates(calendar["end_date"], "calendar end")
        require(calendar["start_date"].le(calendar["end_date"]).all(), "Reversed calendar date interval.")
        for day in DAYS:
            require(calendar[day].isin(["0", "1"]).all(), f"Invalid weekday flag: {day}.")
        base = set(calendar.loc[
            calendar["start_date"].le(date_key)
            & calendar["end_date"].ge(date_key)
            & calendar[weekday].eq("1"),
            "service_id",
        ])

    if exceptions is not None and not exceptions.empty:
        require(REQUIRED_COLUMNS["calendar_dates.txt"].issubset(exceptions.columns), "Missing exception columns.")
        require_key(exceptions, ["service_id", "date"], "calendar exceptions")
        validate_gtfs_dates(exceptions["date"], "calendar exception")
        require(exceptions["exception_type"].isin(["1", "2"]).all(), "Unexpected calendar exception type.")
        relevant = exceptions.loc[exceptions["date"].eq(date_key)]
        additions = set(relevant.loc[relevant["exception_type"].eq("1"), "service_id"])
        removals = set(relevant.loc[relevant["exception_type"].eq("2"), "service_id"])

    return (base | additions) - removals


# Fictional checks only: the actual timetable tables remain unchanged.
example_calendar = pd.DataFrame([{
    "service_id": "regular", **{day: "1" for day in DAYS},
    "start_date": "20261001", "end_date": "20261031",
}])
example_exceptions = pd.DataFrame([
    {"service_id": "regular", "date": "20261013", "exception_type": "2"},
    {"service_id": "special", "date": "20261013", "exception_type": "1"},
])

require(
    active_service_ids(example_calendar, example_exceptions, date(2026, 10, 13)) == {"special"},
    "Exception override check failed.",
)
require(
    active_service_ids(example_calendar, example_exceptions, date(2026, 10, 14)) == {"regular"},
    "Ordinary-date check failed.",
)
require(
    active_service_ids(example_calendar, example_exceptions, date(2026, 11, 1)) == set(),
    "Calendar-boundary check failed.",
)
print("Three calendar-selection self-checks passed.")


Three calendar-selection self-checks passed.


## 4. Build the active-trip lookup

Each source trip is retained only when its `service_id` is active in its own feed. Route details are joined **within that feed** with a many-to-one check. We then add `feed_id`, `trip_key`, `route_key` and `service_key` before stacking feeds.

Replacement buses use the source label already reviewed in notebook 05: `route_short_name` equals `Replacement Bus`, ignoring letter case. The field is deliberately called `is_labelled_replacement_bus`; a feed number or route-type code is not substituted for that explicit rule. We flag these records rather than remove them.

The audit-established expectations for **this exact snapshot and date** are:

| Feed | Active trip records | Labelled replacement-bus trip records |
|---|---:|---:|
| 2 | 2,828 | 52 |
| 3 | 5,044 | 0 |
| 4 | 27,708 | 0 |
| **Total** | **35,580** | **52** |

These constants are verification checks—not invented replacement data. A changed date or source needs reviewed expectations. The output preserves the original trip columns, including optional columns that are absent from some feeds; blank combined optional fields are not filled with invented values.

In [4]:
EXPECTED_ACTIVE_TRIPS = {"2": 2828, "3": 5044, "4": 27708}
EXPECTED_REPLACEMENT_TRIPS = {"2": 52, "3": 0, "4": 0}

parts = []
summary_rows = []
for feed_id in FEED_IDS:
    tables = feed_tables[feed_id]
    trips = tables["trips.txt"]
    routes = tables["routes.txt"]
    calendar = tables["calendar.txt"]
    exceptions = tables["calendar_dates.txt"]

    require_key(trips, ["trip_id"], f"feed {feed_id}/trips")
    require_key(routes, ["route_id"], f"feed {feed_id}/routes")
    require(trips["route_id"].isin(routes["route_id"]).all(), f"Unknown trip-to-route reference in feed {feed_id}.")
    all_defined_services = set(calendar["service_id"]) | set(exceptions["service_id"])
    require(trips["service_id"].isin(all_defined_services).all(), f"Unknown service ID in feed {feed_id}.")

    service_ids = active_service_ids(calendar, exceptions, SERVICE_DATE)
    selected = trips.loc[trips["service_id"].isin(service_ids)].copy()
    selected_before_join = selected.copy(deep=True)
    selected = selected.merge(
        routes[["route_id", "route_short_name", "route_long_name", "route_type"]],
        on="route_id", how="left", validate="many_to_one", indicator=True,
    )
    require(selected["_merge"].eq("both").all(), f"A selected trip has no route in feed {feed_id}.")
    selected = selected.drop(columns="_merge")
    # Every source trip field must survive the route join unchanged.
    pd.testing.assert_frame_equal(
        selected[list(trips.columns)].sort_values("trip_id").reset_index(drop=True),
        selected_before_join.sort_values("trip_id").reset_index(drop=True),
        check_exact=True,
    )

    selected["feed_id"] = pd.Series(feed_id, index=selected.index, dtype="string")
    selected["service_date"] = SERVICE_DATE.isoformat()
    selected["agency_timezone"] = "Australia/Melbourne"
    selected["trip_key"] = feed_id + ":" + selected["trip_id"]
    selected["route_key"] = feed_id + ":" + selected["route_id"]
    selected["service_key"] = feed_id + ":" + selected["service_id"]
    selected["is_labelled_replacement_bus"] = selected["route_short_name"].str.casefold().eq("replacement bus")
    replacement_count = int(selected["is_labelled_replacement_bus"].sum())

    require(len(selected) == EXPECTED_ACTIVE_TRIPS[feed_id], f"Active-trip count differs from audit for feed {feed_id}.")
    require(replacement_count == EXPECTED_REPLACEMENT_TRIPS[feed_id], f"Replacement-trip count differs for feed {feed_id}.")
    parts.append(selected)
    summary_rows.append({
        "feed_id": feed_id,
        "active_trip_records": len(selected),
        "labelled_replacement_trip_records": replacement_count,
    })

active_trips = (
    pd.concat(parts, ignore_index=True)
    .sort_values(["feed_id", "trip_id"], kind="stable")
    .reset_index(drop=True)
)
require(not active_trips.duplicated(["feed_id", "trip_id"]).any(), "Duplicate feed/trip key.")
require(active_trips["trip_key"].is_unique, "Prefixed trip key is not unique.")
require(len(active_trips) == sum(EXPECTED_ACTIVE_TRIPS.values()), "Unexpected combined active-trip count.")
require(active_trips["service_date"].eq(SERVICE_DATE.isoformat()).all(), "Mixed service dates.")
for path, original_hash in input_hashes.items():
    require(sha256_file(path) == original_hash, f"An input changed during this run: {path.name}")

active_trip_summary = pd.DataFrame(summary_rows)
display(active_trip_summary)
display(active_trips[[
    "feed_id", "trip_id", "trip_key", "route_short_name", "service_date",
    "is_labelled_replacement_bus",
]].head(10))

print("ACTIVE TRIP CHECKPOINT PASSED")
print("Service date:", SERVICE_DATE, "(Australia/Melbourne)")
print("Active trip records:", len(active_trips))
print("Labelled replacement-bus trip records:", int(active_trips["is_labelled_replacement_bus"].sum()))
print("Duplicate feed/trip keys: 0")
print("LGA service indicators calculated: False")
print("Input files unchanged: True")
print("Files exported: 0")


,feed_id,active_trip_records,labelled_replacement_trip_records
0,2,2828,52
1,3,5044,0
2,4,27708,0


,feed_id,trip_id,trip_key,route_short_name,service_date,is_labelled_replacement_bus
0,2,02-ALM--70-T5-2201,2:02-ALM--70-T5-2201,Alamein,2026-10-13,False
1,2,02-ALM--70-T5-2203,2:02-ALM--70-T5-2203,Alamein,2026-10-13,False
2,2,02-ALM--70-T5-2205,2:02-ALM--70-T5-2205,Alamein,2026-10-13,False
3,2,02-ALM--70-T5-2207,2:02-ALM--70-T5-2207,Alamein,2026-10-13,False
4,2,02-ALM--70-T5-2209,2:02-ALM--70-T5-2209,Alamein,2026-10-13,False
5,2,02-ALM--70-T5-2211,2:02-ALM--70-T5-2211,Alamein,2026-10-13,False
6,2,02-ALM--70-T5-2212,2:02-ALM--70-T5-2212,Alamein,2026-10-13,False
7,2,02-ALM--70-T5-2213,2:02-ALM--70-T5-2213,Alamein,2026-10-13,False
8,2,02-ALM--70-T5-2214,2:02-ALM--70-T5-2214,Alamein,2026-10-13,False
9,2,02-ALM--70-T5-2215,2:02-ALM--70-T5-2215,Alamein,2026-10-13,False


ACTIVE TRIP CHECKPOINT PASSED
Service date: 2026-10-13 (Australia/Melbourne)
Active trip records: 35580
Labelled replacement-bus trip records: 52
Duplicate feed/trip keys: 0
LGA service indicators calculated: False
Input files unchanged: True
Files exported: 0


## Part A checkpoint

The expected lookup has **35,580 active trip records**: feed 2 = 2,828; feed 3 = 5,044; feed 4 = 27,708. It includes **52 explicitly labelled replacement-bus trips**, not 35,580 vehicles or trips confined to our LGAs.

Part A's checks must pass before continuing. Part B will read all three stop-time files in chunks, retain only these active trips, and determine each complete trip's final stop **before** applying spatial or time filters.


# Part B — Date-specific stop events and LGA summaries
## 5. Load the reviewed stop lookup and full LGA inventory

The spatial manifest verifies your own CSV bytes; we do not substitute another computer's CSV fingerprint.
The crosswalk supplies all **31 LGAs**, including any area/category/window with no counted events.
Blank LGA codes for outside records are preserved. A missing lookup key is an error, not a zero-service area.

We track additional upstream files so that the export stage can confirm none was overwritten.

In [5]:
import numpy as np
from contextlib import contextmanager
from tempfile import SpooledTemporaryFile, TemporaryDirectory
from time import perf_counter
from datetime import timezone
import io
import shutil

LGA_PATH = ROOT / "data/interim/geography/lga_crosswalk_2025.csv"
GEO_REPORT_PATH = ROOT / "data/interim/geography/geography_pipeline_report.json"
for path in [LGA_PATH, GEO_REPORT_PATH]:
    require(path.is_file(), f"Missing earlier geography output: {path}")
    input_hashes[path] = sha256_file(path)
geo_report = read_json_object(GEO_REPORT_PATH)
geo_output_hashes = {
    key.replace("\\", "/"): value
    for key, value in geo_report.get("output_sha256", {}).items()
}
require(
    geo_output_hashes.get(LGA_PATH.relative_to(ROOT).as_posix()) == input_hashes[LGA_PATH],
    "The reviewed LGA crosswalk differs from its geography report.",
)

stops_lookup = pd.read_csv(STOP_PATH, dtype="string", keep_default_na=False)
STOP_KEYS = ["feed_id", "stop_id"]
STOP_COLUMNS = [
    *STOP_KEYS, "stop_key", "stop_name", "lga_code_2025",
    "abs_lga_name", "spatial_match_status", "near_boundary_10m",
]
require(set(STOP_COLUMNS).issubset(stops_lookup.columns), "Stop lookup schema differs from notebook 06.")
require_key(stops_lookup, STOP_KEYS, "stop lookup")
require(len(stops_lookup) == 25031, "Unexpected stop/platform inventory size.")
require(set(stops_lookup.feed_id) == set(FEED_IDS), "Stop lookup has a different feed scope.")
near_text = stops_lookup["near_boundary_10m"].str.lower()
require(near_text.isin(["true", "false"]).all(), "Unexpected near-boundary flag.")
stops_lookup["near_boundary_10m"] = near_text.eq("true")
require(
    stops_lookup.spatial_match_status.value_counts().to_dict()
    == {"matched_within": 21426, "outside_selected_lgas": 3605},
    "Spatial outcomes differ from the reviewed snapshot.",
)
require(stops_lookup.stop_key.eq(stops_lookup.feed_id + ":" + stops_lookup.stop_id).all(),
        "Stop namespace is inconsistent.")
lga_inventory = pd.read_csv(LGA_PATH, dtype="string", keep_default_na=False)
lga_inventory = lga_inventory[["lga_code_2025", "abs_lga_name", "rental_area_name"]].copy()
require_key(lga_inventory, ["lga_code_2025"], "LGA inventory")
require(len(lga_inventory) == 31, "Expected the same 31 LGAs.")
inside = stops_lookup.spatial_match_status.eq("matched_within")
require(stops_lookup.loc[inside, "lga_code_2025"].isin(lga_inventory.lga_code_2025).all(),
        "A stop assignment uses an unknown LGA.")
require(stops_lookup.loc[~inside, "lga_code_2025"].eq("").all(),
        "Outside records must not have forced LGA assignments.")
official = lga_inventory.set_index("lga_code_2025")["abs_lga_name"]
require(
    stops_lookup.loc[inside, "abs_lga_name"].eq(
        stops_lookup.loc[inside, "lga_code_2025"].map(official)
    ).all(), "Stop names/codes disagree with the reviewed crosswalk.",
)

WINDOWS = {
    "service_day": (None, None),
    "am_peak_0700_0900": (7 * 3600, 9 * 3600),
}
CATEGORY_FEED = {
    "train_not_labelled_replacement": "2",
    "labelled_replacement_bus": "2",
    "tram": "3",
    "myki_bus": "4",
}
active_for_service = active_trips.copy(deep=True)
active_for_service["service_category"] = active_for_service.feed_id.map({
    "2": "train_not_labelled_replacement", "3": "tram", "4": "myki_bus",
})
active_for_service.loc[
    active_for_service.is_labelled_replacement_bus, "service_category"
] = "labelled_replacement_bus"
require(
    active_for_service.is_labelled_replacement_bus.eq(False).all()
    or active_for_service.loc[
        active_for_service.is_labelled_replacement_bus, "feed_id"
    ].eq("2").all(), "Unexpected replacement-service scope.",
)
print("SERVICE INPUTS READY")
print("Selected stop/platform records:", len(stops_lookup))
print("LGA inventory:", len(lga_inventory))
print("Service categories:", len(CATEGORY_FEED), "| Time windows:", len(WINDOWS))
print("The 10 m boundary flag is retained as a diagnostic, not reassignment.")

SERVICE INPUTS READY
Selected stop/platform records: 25031
LGA inventory: 31
Service categories: 4 | Time windows: 2
The 10 m boundary flag is retained as a diagnostic, not reassignment.


## 6. Define exactly what we count

An event is one `(feed_id, trip_id, numeric stop_sequence)` in an **active** trip.
The counted subset must have:
1. ordinary pickup (`pickup_type` = `0`, including blank under GTFS defaults);
2. a later stop in the **complete trip**, not merely a later stop inside the same LGA/window;
3. a `matched_within` assignment to one selected LGA.

Final trip stops, no-pickup records (`1`), and arrangement-required records (`2`/`3`) are retained in separate exclusion counts. The terminal rule is our conservative metric choice, not a new GTFS validity rule. Linked-trip/block continuation is not modelled. Drop-off codes are retained but we do not solve destination-specific alighting or transfers.

Replacement buses form a separate category. `train_not_labelled_replacement` literally means the train operational feed with the explicit replacement label removed; it is not a universal mode classifier for other feeds.

For each window, every active event receives **one exclusive outcome**, in this order: terminal, no pickup, arrangement required, unassigned location, counted. An outside terminal is therefore counted in the terminal exclusion, not counted twice.

**Read this function first:** `classify_active_events()` computes the complete-trip maximum stop sequence **before** selecting a time window or an LGA.
GTFS reference: https://gtfs.org/documentation/schedule/reference/#stop_timestxt

In [6]:
RAW_EVENT_COLUMNS = [
    "trip_id", "stop_id", "stop_sequence", "arrival_time",
    "departure_time", "pickup_type", "drop_off_type",
]
EVENT_KEYS = ["feed_id", "trip_id", "stop_sequence"]
EXPECTED_SCAN_ROWS = {"2": 688380, "3": 2568892, "4": 8589911}
CHUNK_ROWS = 100_000
OUTCOMES = [
    "terminal_trip_record", "no_pickup", "arrangement_required",
    "unassigned_location", "counted_regular_pickup",
]


def seconds_from_gtfs(values: pd.Series) -> pd.Series:
    """Parse the supplied times; do not convert 25:00 to 01:00."""
    text = values.astype("string")
    unique = pd.Series(text.unique(), dtype="string")
    require(unique.notna().all() and unique.str.fullmatch(r"[0-9]{1,2}:[0-5][0-9]:[0-5][0-9]").all(),
            "Blank or invalid GTFS time; do not fill/interpolate it.")
    split = unique.str.split(":", expand=True).astype("int64")
    seconds = split[0] * 3600 + split[1] * 60 + split[2]
    return text.map(dict(zip(unique, seconds))).astype("int64")


def in_window(events: pd.DataFrame, window: str) -> pd.Series:
    require(window in WINDOWS, f"Unknown time window: {window}")
    start, end = WINDOWS[window]
    if start is None:
        return pd.Series(True, index=events.index)
    return events.departure_seconds.ge(start) & events.departure_seconds.lt(end)


def classify_active_events(raw_events, selected_trips, stop_lookup):
    """Classify a complete feed's active stop sequences, independent of row order."""
    require(len(raw_events) > 0, "No active stop-time rows found.")
    require(set(RAW_EVENT_COLUMNS + ["feed_id", "source_line_number"]).issubset(raw_events.columns),
            "Incomplete active-event schema.")
    events = raw_events.copy()
    require(events.feed_id.nunique() == 1, "Process each operational feed separately.")
    feed_id = str(events.feed_id.iloc[0])
    require(selected_trips.feed_id.eq(feed_id).all(), "Mixed trip namespaces.")
    require_key(selected_trips, ["feed_id", "trip_id"], "selected trips")

    seq_text = events.stop_sequence.astype("string")
    require(seq_text.str.fullmatch(r"[0-9]+").all(), "Stop sequence must be a non-negative integer.")
    events["stop_sequence"] = pd.to_numeric(seq_text, errors="raise").astype("int64")
    require(not events.duplicated(EVENT_KEYS).any(), "Duplicate event key, including across read chunks.")
    require(set(events.trip_id) == set(selected_trips.trip_id), "Missing or unexpected active trip stop times.")
    require(events.stop_id.ne("").all(), "Empty stop identifier.")
    events["arrival_seconds"] = seconds_from_gtfs(events.arrival_time)
    events["departure_seconds"] = seconds_from_gtfs(events.departure_time)
    require(events.arrival_seconds.le(events.departure_seconds).all(), "Arrival is later than departure at a stop.")
    for field in ["pickup_type", "drop_off_type"]:
        require(events[field].astype("string").isin(["", "0", "1", "2", "3"]).all(),
                f"Unexpected {field} code.")
        events[field + "_effective"] = events[field].replace("", "0")

    # This maximum uses the FULL active trip, including stops outside our LGAs/window.
    by_trip = events.groupby("trip_id", sort=False)
    events["trip_last_sequence"] = by_trip.stop_sequence.transform("max")
    events["has_onward_stop"] = events.stop_sequence.lt(events.trip_last_sequence)
    terminals = events.loc[~events.has_onward_stop]
    require(len(terminals) == len(selected_trips) and terminals.trip_id.is_unique,
            "Expected exactly one final sequence per active trip.")
    # Independent max check after sorting. No contiguous sequence numbering is assumed.
    sorted_last = events.sort_values(["trip_id", "stop_sequence"]).groupby("trip_id", sort=False).tail(1)
    require(set(zip(sorted_last.trip_id, sorted_last.stop_sequence))
            == set(zip(terminals.trip_id, terminals.stop_sequence)), "Terminal detection disagrees.")

    trip_fields = ["feed_id", "trip_id", "trip_key", "route_id", "route_key",
                   "service_category", "is_labelled_replacement_bus"]
    before_rows = len(events)
    events = events.merge(selected_trips[trip_fields], on=["feed_id", "trip_id"],
                          how="left", validate="many_to_one", indicator="_trip_join")
    require(events._trip_join.eq("both").all(), "Unmatched active trip.")
    events = events.drop(columns="_trip_join")
    events = events.merge(stop_lookup[STOP_COLUMNS], on=STOP_KEYS,
                          how="left", validate="many_to_one", indicator="_stop_join")
    require(len(events) == before_rows and events._stop_join.eq("both").all(),
            "Missing or ambiguous feed/stop lookup; outside status is different from a missing key.")
    events = events.drop(columns="_stop_join")
    events["regular_pickup"] = events.pickup_type_effective.eq("0")
    assigned = events.spatial_match_status.eq("matched_within")
    events["selection_outcome"] = np.select(
        [
            (~events.has_onward_stop).to_numpy(dtype=bool),
            events.pickup_type_effective.eq("1").to_numpy(dtype=bool),
            events.pickup_type_effective.isin(["2", "3"]).to_numpy(dtype=bool),
            (~assigned).to_numpy(dtype=bool),
        ],
        OUTCOMES[:-1], default="counted_regular_pickup",
    )
    counted = events.selection_outcome.eq("counted_regular_pickup")
    require(counted.eq(events.has_onward_stop & events.regular_pickup & assigned).all(),
            "Selection outcome differs from the declared rule.")
    require(events.loc[counted, "lga_code_2025"].ne("").all(), "Counted event has no LGA.")
    return events


@contextmanager
def open_stop_time_chunks(feed_id):
    """Read one nested ZIP at a time; spill compressed bytes to temporary disk."""
    member = f"{feed_id}/google_transit.zip"
    with ZipFile(GTFS_PATH) as outer, SpooledTemporaryFile(max_size=32 * 1024 * 1024) as spool:
        with outer.open(member) as source:
            shutil.copyfileobj(source, spool, length=1024 * 1024)
        spool.seek(0)
        with ZipFile(spool) as inner:
            require("frequencies.txt" not in inner.namelist(), "Frequency-based trips need a different method.")
            with inner.open("stop_times.txt") as handle:
                header = pd.read_csv(handle, nrows=0).columns.tolist()
            require(set(RAW_EVENT_COLUMNS).issubset(header), "Missing audited stop-time fields.")
            unsupported = {
                "location_id", "location_group_id", "start_pickup_drop_off_window",
                "end_pickup_drop_off_window", "pickup_booking_rule_id",
                "drop_off_booking_rule_id", "continuous_pickup", "continuous_drop_off",
            }
            require(not unsupported.intersection(header),
                    "Flexible/continuous-service fields need a reviewed extension to this fixed-stop method.")
            with inner.open("stop_times.txt") as handle:
                yield pd.read_csv(
                    handle, usecols=RAW_EVENT_COLUMNS, dtype="string",
                    keep_default_na=False, encoding="utf-8-sig", chunksize=CHUNK_ROWS,
                )

## 7. Summarise events without confusing trips, visits and routes

For one LGA/window/category:
- `eligible_departure_events`: counted stop-sequence records; one trip can contribute several.
- `unique_trip_records`: distinct feed-prefixed trips with at least one counted event in that LGA/window.
- `served_stop_records`: distinct feed-prefixed stops with counted events, not physical stations.
- `unique_route_records`: distinct feed-prefixed route IDs, not necessarily passenger-facing route brands.

Every LGA/category/window combination appears in the final table, including zero-count combinations.
The network summary is calculated independently from the counted events: **never sum LGA trip counts to obtain network trips**. A trip can serve several LGAs.

The complete service-day total retains 24:00+ records. The morning window tests the *unwrapped* departure seconds. No average waiting time, headway, capacity or suitability score is calculated.

In [7]:
COUNT_FIELDS = [
    "eligible_departure_events", "unique_trip_records", "served_stop_records",
    "unique_route_records", "near_boundary_events", "extended_time_events",
]
LGA_GROUP = ["lga_code_2025", "feed_id", "service_category", "window"]


def summarize_events(events):
    """Return small analytical tables; never retain all feeds' raw events together."""
    lga_parts, stop_parts, membership_parts = [], [], []
    audit_rows, network_rows, sample_parts = [], [], []
    sample_columns = [
        *EVENT_KEYS, "source_line_number", "stop_id", "stop_key", "trip_key",
        "route_key", "service_category", "arrival_time", "departure_time",
        "departure_seconds", "pickup_type", "drop_off_type", "trip_last_sequence",
        "has_onward_stop", "lga_code_2025", "abs_lga_name", "spatial_match_status",
        "near_boundary_10m", "selection_outcome",
    ]
    categories = events[["feed_id", "service_category"]].drop_duplicates()
    for window in WINDOWS:
        window_events = events.loc[in_window(events, window)].copy()
        for cat in categories.itertuples(index=False):
            subset = window_events.loc[window_events.service_category.eq(cat.service_category)]
            counts = subset.selection_outcome.value_counts()
            partition = {name: int(counts.get(name, 0)) for name in OUTCOMES}
            require(sum(partition.values()) == len(subset), "Exclusive selection outcomes do not reconcile.")
            audit_rows.append({
                "feed_id": cat.feed_id, "service_category": cat.service_category, "window": window,
                "active_stop_time_rows_in_window": len(subset), **partition,
            })
            chosen = subset.loc[subset.selection_outcome.eq("counted_regular_pickup")]
            network_rows.append({
                "feed_id": cat.feed_id, "service_category": cat.service_category, "window": window,
                "eligible_departure_events": len(chosen),
                "unique_trip_records": chosen.trip_key.nunique(),
                "served_stop_records": chosen.stop_key.nunique(),
                "unique_route_records": chosen.route_key.nunique(),
                "near_boundary_events": int(chosen.near_boundary_10m.sum()),
                "extended_time_events": int(chosen.departure_seconds.ge(86400).sum()),
            })
            # Traceable examples, NOT a substitute for the full-table calculation.
            for outcome in OUTCOMES:
                sample = subset.loc[subset.selection_outcome.eq(outcome), sample_columns].head(3).copy()
                if not sample.empty:
                    sample["window"] = window
                    sample_parts.append(sample)

        chosen = window_events.loc[
            window_events.selection_outcome.eq("counted_regular_pickup")
        ].copy()
        chosen["window"] = window
        chosen["extended_time_event"] = chosen.departure_seconds.ge(86400)
        if not chosen.empty:
            lga_parts.append(chosen.groupby(LGA_GROUP, sort=True).agg(
                eligible_departure_events=("trip_key", "size"),
                unique_trip_records=("trip_key", "nunique"),
                served_stop_records=("stop_key", "nunique"),
                unique_route_records=("route_key", "nunique"),
                near_boundary_events=("near_boundary_10m", "sum"),
                extended_time_events=("extended_time_event", "sum"),
            ).reset_index())
            stop_parts.append(chosen.groupby(
                ["feed_id", "stop_id", "stop_key", "lga_code_2025", "service_category", "window"],
                sort=True,
            ).agg(
                eligible_departure_events=("trip_key", "size"),
                unique_trip_records=("trip_key", "nunique"),
                unique_route_records=("route_key", "nunique"),
                first_departure_seconds=("departure_seconds", "min"),
                last_departure_seconds=("departure_seconds", "max"),
                near_boundary_10m=("near_boundary_10m", "first"),
            ).reset_index())
            membership_parts.append(chosen[
                ["lga_code_2025", "feed_id", "service_category", "window", "trip_key", "route_key"]
            ].drop_duplicates())

    trip_scope = events.groupby(
        ["feed_id", "trip_key", "route_key", "service_category"], sort=True
    ).agg(
        complete_stop_time_rows=("stop_sequence", "size"),
        first_sequence=("stop_sequence", "min"), final_sequence=("stop_sequence", "max"),
        first_departure_seconds=("departure_seconds", "min"),
        last_departure_seconds=("departure_seconds", "max"),
    ).reset_index()
    # Count all source events, including excluded/outside records, in the trip-level audit.
    require(int(trip_scope.complete_stop_time_rows.sum()) == len(events), "Trip-scope row counts differ.")
    return {
        "lga": pd.concat(lga_parts, ignore_index=True) if lga_parts else pd.DataFrame(columns=LGA_GROUP + COUNT_FIELDS),
        "stops": pd.concat(stop_parts, ignore_index=True) if stop_parts else pd.DataFrame(),
        "memberships": pd.concat(membership_parts, ignore_index=True) if membership_parts else pd.DataFrame(),
        "filter_audit": pd.DataFrame(audit_rows),
        "network": pd.DataFrame(network_rows),
        "examples": pd.concat(sample_parts, ignore_index=True) if sample_parts else pd.DataFrame(),
        "trip_scope": trip_scope,
    }


def scan_service_feed(feed_id):
    """Scan all rows once, retaining only active trips in this feed."""
    selected = active_for_service.loc[active_for_service.feed_id.eq(feed_id)].copy()
    wanted = set(selected.trip_id)
    started = perf_counter()
    pieces, total_rows, retained = [], 0, 0
    print(f"Scanning feed {feed_id} stop_times.txt...", flush=True)
    with open_stop_time_chunks(feed_id) as chunks:
        for number, chunk in enumerate(chunks, start=1):
            part = chunk.loc[chunk.trip_id.isin(wanted)].copy()
            # pandas chunk indexes are global here; use explicit offsets instead of relying on that.
            positions = np.flatnonzero(chunk.trip_id.isin(wanted).to_numpy(dtype=bool))
            part["source_line_number"] = total_rows + positions + 2  # header is line 1
            part["feed_id"] = pd.Series(feed_id, index=part.index, dtype="string")
            if not part.empty:
                pieces.append(part)
                retained += len(part)
            total_rows += len(chunk)
            if number == 1 or number % 10 == 0:
                print(f"  Read {total_rows:,} source rows; retained {retained:,} active-trip rows.", flush=True)
    require(total_rows == EXPECTED_SCAN_ROWS[feed_id], f"Source row count changed in feed {feed_id}.")
    require(pieces, f"No active records in feed {feed_id}.")
    raw_active = pd.concat(pieces, ignore_index=True)
    del pieces
    events = classify_active_events(raw_active, selected, stops_lookup)
    del raw_active
    results = summarize_events(events)
    results["scan"] = {
        "feed_id": feed_id, "source_stop_time_rows_scanned": total_rows,
        "active_trip_records": len(selected), "active_stop_time_rows": len(events),
        "terminal_trip_records": int((~events.has_onward_stop).sum()),
        "after_midnight_active_rows": int(events.departure_seconds.ge(86400).sum()),
        "minimum_departure_seconds": int(events.departure_seconds.min()),
        "maximum_departure_seconds": int(events.departure_seconds.max()),
    }
    print(f"Feed {feed_id} done: {total_rows:,} rows scanned; {len(events):,} active rows; "
          f"{perf_counter()-started:.1f}s.", flush=True)
    display(results["filter_audit"])
    # Only small aggregates, trip memberships and trace examples survive this function.
    return results

## 8. Run small policy tests before scanning the real timetable

These examples deliberately include repeated visits to one stop, non-consecutive sequence numbers,
a terminal outside the area and after the morning window, no-pickup records, arrangement-required
records, and times beyond midnight. They are not added to the real data or the pytest suite.

In [8]:
def run_service_self_checks():
    checks = []
    fixture_rows = [
        # One trip visits A twice, B at 09:00, then ends outside at 25:00.
        ("T", "A", "1", "07:00:00", "0"),
        ("T", "A", "3", "07:30:00", ""),  # blank means regular pickup
        ("T", "B", "8", "09:00:00", "0"),
        ("T", "C", "10", "25:00:00", "0"),
        ("N", "A", "1", "08:00:00", "1"),
        ("N", "C", "2", "08:20:00", "0"),
        ("R", "A", "0", "08:10:00", "3"),
        ("R", "C", "4", "08:30:00", "0"),
    ]
    raw = pd.DataFrame(fixture_rows, columns=["trip_id", "stop_id", "stop_sequence", "departure_time", "pickup_type"])
    raw["arrival_time"] = raw.departure_time
    raw["drop_off_type"] = "0"
    raw["feed_id"] = "4"
    raw["source_line_number"] = np.arange(len(raw)) + 2
    trips = pd.DataFrame({
        "feed_id": ["4"] * 3, "trip_id": ["T", "N", "R"],
        "trip_key": ["4:T", "4:N", "4:R"], "route_id": ["R1"] * 3,
        "route_key": ["4:R1"] * 3, "service_category": ["myki_bus"] * 3,
        "is_labelled_replacement_bus": [False] * 3,
    })
    lookup = pd.DataFrame({
        "feed_id": ["4"] * 3, "stop_id": ["A", "B", "C"],
        "stop_key": ["4:A", "4:B", "4:C"], "stop_name": ["Alpha", "Beta", "Outside"],
        "lga_code_2025": ["20001", "20002", ""],
        "abs_lga_name": ["Area A", "Area B", ""],
        "spatial_match_status": ["matched_within", "matched_within", "outside_selected_lgas"],
        "near_boundary_10m": [False, True, False],
    })
    events = classify_active_events(raw, trips, lookup)
    checks.append("complete-trip terminal rule")
    require(events.loc[events.trip_id.eq("T") & events.stop_id.eq("B"), "has_onward_stop"].all(),
            "Onward stop outside the study area/time window was lost.")
    require(int(events.selection_outcome.eq("counted_regular_pickup").sum()) == 3, "Wrong known count.")
    require(int(events.pickup_type.eq("").sum()) == 1, "Fixture blank pickup lost.")
    checks.append("blank pickup defaults and known result")
    require(events.selection_outcome.value_counts().get("no_pickup", 0) == 1
            and events.selection_outcome.value_counts().get("arrangement_required", 0) == 1,
            "Pickup exclusion policy failed.")
    checks.append("no-pickup and arrangement separation")
    counted_peak = events.loc[in_window(events, "am_peak_0700_0900")
                             & events.selection_outcome.eq("counted_regular_pickup")]
    require(len(counted_peak) == 2, "07:00 inclusion / 09:00 exclusion failed.")
    require(seconds_from_gtfs(pd.Series(["25:15:30"])).iloc[0] == 90930,
            "Extended service-day time wrapped incorrectly.")
    checks.append("time-window edges and extended times")
    summaries = summarize_events(events)
    net_day = summaries["network"].query("window == 'service_day'").iloc[0]
    require(net_day.eligible_departure_events == 3 and net_day.unique_trip_records == 1,
            "Repeated stop visits were confused with different trips.")
    require(int(summaries["lga"].query("window == 'service_day'").unique_trip_records.sum()) == 2,
            "Cross-LGA trip-membership fixture failed.")
    checks.append("events versus trips and nonadditive LGA trips")
    shuffled = classify_active_events(raw.sample(frac=1, random_state=7), trips, lookup)
    fields = EVENT_KEYS + ["trip_last_sequence", "selection_outcome"]
    pd.testing.assert_frame_equal(events[fields].sort_values(EVENT_KEYS).reset_index(drop=True),
                                  shuffled[fields].sort_values(EVENT_KEYS).reset_index(drop=True))
    checks.append("row-order independence")

    def reject(call, expected_message):
        try:
            call()
        except ValueError as error:
            require(expected_message in str(error), f"Unexpected rejection: {error}")
        else:
            raise ValueError("A deliberately invalid fixture was accepted.")

    duplicate = pd.concat([raw, raw.iloc[[0]]], ignore_index=True)
    reject(lambda: classify_active_events(duplicate, trips, lookup), "Duplicate event key")
    checks.append("duplicate event keys including chunk boundaries")
    reject(lambda: classify_active_events(raw, trips, lookup.loc[lookup.stop_id.ne("C")]),
           "Missing or ambiguous feed/stop lookup")
    checks.append("unknown stop lookup is not outside")
    reject(lambda: seconds_from_gtfs(pd.Series(["07:60:00"])), "invalid GTFS time")
    checks.append("malformed time rejection")
    other_lookup = lookup.copy()
    other_lookup["feed_id"] = "3"
    other_lookup["stop_key"] = "3:" + other_lookup.stop_id
    other_lookup.loc[other_lookup.stop_id.eq("A"), "lga_code_2025"] = "29999"
    both_lookups = pd.concat([lookup, other_lookup], ignore_index=True)
    namespaced = classify_active_events(raw, trips, both_lookups)
    require(namespaced.loc[namespaced.stop_id.eq("A"), "lga_code_2025"].eq("20001").all(),
            "Cross-feed identifiers were incorrectly joined.")
    checks.append("feed-specific namespaces")
    require(len(checks) == 10, "Unexpected self-check count.")
    print("Ten service-policy self-check groups passed.")
    return checks

service_self_checks = run_service_self_checks()
feed_results = {}  # Scan outputs will be small aggregates, not all raw events.

Ten service-policy self-check groups passed.


## 9. Scan feed 2 — Metropolitan train operational feed

The reader scans the full source file in chunks of 100,000 rows, but only keeps the
active trips for 13 October 2026. It processes one feed at a time. Allow it to finish;
do not repeatedly click Run while progress messages are updating.

Final sequences are determined after all chunks for this feed have been collected.
No stop outside the LGAs is removed until that step is complete.

In [9]:
feed_results["2"] = scan_service_feed("2")

Scanning feed 2 stop_times.txt...
  Read 100,000 source rows; retained 6,471 active-trip rows.
Feed 2 done: 688,380 rows scanned; 49,389 active rows; 1.6s.


,feed_id,service_category,window,active_stop_time_rows_in_window,terminal_trip_record,no_pickup,arrangement_required,unassigned_location,counted_regular_pickup
0,2,train_not_labelled_replacement,service_day,48857,2776,0,0,0,46081
1,2,labelled_replacement_bus,service_day,532,52,0,0,0,480
2,2,train_not_labelled_replacement,am_peak_0700_0900,7762,415,0,0,0,7347
3,2,labelled_replacement_bus,am_peak_0700_0900,0,0,0,0,0,0


## 10. Scan feed 3 — Metropolitan tram

The reader scans the full source file in chunks of 100,000 rows, but only keeps the
active trips for 13 October 2026. It processes one feed at a time. Allow it to finish;
do not repeatedly click Run while progress messages are updating.

Final sequences are determined after all chunks for this feed have been collected.
No stop outside the LGAs is removed until that step is complete.

In [10]:
feed_results["3"] = scan_service_feed("3")

Scanning feed 3 stop_times.txt...
  Read 100,000 source rows; retained 12,688 active-trip rows.
  Read 1,000,000 source rows; retained 88,250 active-trip rows.
  Read 2,000,000 source rows; retained 182,185 active-trip rows.
Feed 3 done: 2,568,892 rows scanned; 232,760 active rows; 5.9s.


,feed_id,service_category,window,active_stop_time_rows_in_window,terminal_trip_record,no_pickup,arrangement_required,unassigned_location,counted_regular_pickup
0,3,tram,service_day,232760,5044,0,0,0,227716
1,3,tram,am_peak_0700_0900,29209,567,0,0,0,28642


## 11. Scan feed 4 — Myki bus

The reader scans the full source file in chunks of 100,000 rows, but only keeps the
active trips for 13 October 2026. It processes one feed at a time. Allow it to finish;
do not repeatedly click Run while progress messages are updating.

Final sequences are determined after all chunks for this feed have been collected.
No stop outside the LGAs is removed until that step is complete.

In [11]:
feed_results["4"] = scan_service_feed("4")

Scanning feed 4 stop_times.txt...
  Read 100,000 source rows; retained 8,173 active-trip rows.
  Read 1,000,000 source rows; retained 78,093 active-trip rows.
  Read 2,000,000 source rows; retained 157,676 active-trip rows.
  Read 3,000,000 source rows; retained 240,963 active-trip rows.
  Read 4,000,000 source rows; retained 348,111 active-trip rows.
  Read 5,000,000 source rows; retained 501,288 active-trip rows.
  Read 6,000,000 source rows; retained 672,088 active-trip rows.
  Read 7,000,000 source rows; retained 776,084 active-trip rows.
  Read 8,000,000 source rows; retained 912,144 active-trip rows.
Feed 4 done: 8,589,911 rows scanned; 1,029,597 active rows; 55.5s.


,feed_id,service_category,window,active_stop_time_rows_in_window,terminal_trip_record,no_pickup,arrangement_required,unassigned_location,counted_regular_pickup
0,4,myki_bus,service_day,1029597,27708,30,0,100855,901004
1,4,myki_bus,am_peak_0700_0900,140739,3731,0,0,14065,122943


## 12. Build the complete LGA table and independently reconcile the counts

The final grain is **one LGA × one service category × one time window × one service date**:
31 × 4 × 2 = **248 rows**. Only the three audited feeds are included.

We check counts using three different views: event aggregates, distinct trip/LGA memberships, and stop-level totals. Network trip counts are distinct across the study area; LGA trip counts are intentionally not additive across areas.

In [12]:
require(set(feed_results) == set(FEED_IDS), "Complete all three feed scans before combining results.")


def combine_result(name):
    return pd.concat([feed_results[f][name] for f in FEED_IDS], ignore_index=True)


lga_nonzero = combine_result("lga")
stop_service = combine_result("stops")
trip_lga_memberships = combine_result("memberships")
filter_audit = combine_result("filter_audit")
network_service = combine_result("network")
trace_examples = combine_result("examples")
trip_stop_scope = combine_result("trip_scope")
scan_summary = pd.DataFrame([feed_results[f]["scan"] for f in FEED_IDS])

require(int(scan_summary.source_stop_time_rows_scanned.sum()) == 11847183, "Incomplete source scan.")
require(int(scan_summary.active_trip_records.sum()) == len(active_for_service), "Active-trip reconciliation failed.")
require(int(trip_stop_scope.complete_stop_time_rows.sum()) == int(scan_summary.active_stop_time_rows.sum()),
        "Active stop-time rows do not reconcile.")
require_key(trip_stop_scope, ["trip_key"], "complete-trip scope")
require(set(trip_stop_scope.trip_key) == set(active_for_service.trip_key), "A complete trip is missing.")
require_key(lga_nonzero, LGA_GROUP, "nonzero LGA aggregates")
require_key(trip_lga_memberships, LGA_GROUP + ["trip_key"], "trip/LGA/window memberships")

category_table = pd.DataFrame([
    {"service_category": category, "feed_id": feed}
    for category, feed in CATEGORY_FEED.items()
])
grid = lga_inventory.merge(category_table, how="cross").merge(
    pd.DataFrame({"window": list(WINDOWS)}), how="cross"
)
lga_service = grid.merge(lga_nonzero, on=LGA_GROUP, how="left", validate="one_to_one")
for col in COUNT_FIELDS:
    lga_service[col] = lga_service[col].fillna(0).astype("int64")
lga_service["eligible_events_excluding_near_boundary"] = (
    lga_service.eligible_departure_events - lga_service.near_boundary_events
)
lga_service = lga_service.sort_values(
    ["window", "lga_code_2025", "service_category"], kind="stable"
).reset_index(drop=True)
require(len(lga_service) == 248, "Incomplete LGA/category/window grid.")
require_key(lga_service, LGA_GROUP, "complete LGA service table")
require(lga_service[COUNT_FIELDS].ge(0).all().all(), "Negative summary count.")
for col in COUNT_FIELDS[1:]:
    require(lga_service[col].le(lga_service.eligible_departure_events).all(),
            f"{col} exceeds the counted event total.")

# Independent group counts from the saved trip memberships and stop aggregates.
reference = lga_service.set_index(LGA_GROUP)
membership_counts = trip_lga_memberships.groupby(LGA_GROUP).trip_key.nunique().reindex(reference.index, fill_value=0)
stop_event_counts = stop_service.groupby(LGA_GROUP).eligible_departure_events.sum().reindex(reference.index, fill_value=0)
stop_record_counts = stop_service.groupby(LGA_GROUP).stop_key.nunique().reindex(reference.index, fill_value=0)
require(np.array_equal(reference.unique_trip_records.to_numpy(), membership_counts.to_numpy()),
        "LGA trip counts differ from independent membership counts.")
require(np.array_equal(reference.eligible_departure_events.to_numpy(), stop_event_counts.to_numpy()),
        "LGA event counts differ from summed stop-level events.")
require(np.array_equal(reference.served_stop_records.to_numpy(), stop_record_counts.to_numpy()),
        "LGA stop counts differ from independent stop identities.")
net_keys = ["feed_id", "service_category", "window"]
network_reference = network_service.set_index(net_keys)
lga_event_sums = lga_service.groupby(net_keys).eligible_departure_events.sum().reindex(network_reference.index, fill_value=0)
network_trip_counts = trip_lga_memberships.groupby(net_keys).trip_key.nunique().reindex(network_reference.index, fill_value=0)
require(np.array_equal(network_reference.eligible_departure_events, lga_event_sums),
        "An eligible event was lost or counted in multiple LGAs.")
require(np.array_equal(network_reference.unique_trip_records, network_trip_counts),
        "Network trips were incorrectly summed across LGAs.")
audit_reference = filter_audit.set_index(net_keys)
require(np.array_equal(
    audit_reference.counted_regular_pickup.reindex(network_reference.index),
    network_reference.eligible_departure_events,
), "Filter-audit counted events disagree.")
require(filter_audit[OUTCOMES].sum(axis=1).eq(filter_audit.active_stop_time_rows_in_window).all(),
        "Filter audit is not an exhaustive, exclusive partition.")
require(
    int(filter_audit.loc[filter_audit.window.eq("service_day"), "terminal_trip_record"].sum())
    == len(active_for_service), "Full service day must exclude one terminal per active trip.",
)

all_day = lga_service.loc[lga_service.window.eq("service_day")].set_index(["lga_code_2025", "service_category"])
morning = lga_service.loc[lga_service.window.eq("am_peak_0700_0900")].set_index(["lga_code_2025", "service_category"])
require(morning[COUNT_FIELDS].le(all_day[COUNT_FIELDS]).all().all(), "Morning is not a subset of service day.")
require(morning.extended_time_events.eq(0).all(), "Extended times were incorrectly wrapped into the morning window.")

for table in [lga_service, stop_service, trip_lga_memberships, filter_audit,
              network_service, trace_examples, trip_stop_scope, scan_summary]:
    table["service_date"] = SERVICE_DATE.isoformat()
    table["agency_timezone"] = "Australia/Melbourne"

print("SERVICE RECONCILIATION PASSED")
print("Source stop-time rows scanned:", int(scan_summary.source_stop_time_rows_scanned.sum()))
print("Active-trip stop-time rows:", int(scan_summary.active_stop_time_rows.sum()))
print("LGA/category/window rows:", len(lga_service))
print("Membership, stop-level and network totals match.")
display(network_service[[
    "service_category", "window", "eligible_departure_events", "unique_trip_records",
    "served_stop_records", "unique_route_records", "extended_time_events",
]])
display(lga_service.loc[
    lga_service.window.eq("am_peak_0700_0900") &
    lga_service.rental_area_name.isin(["Melbourne", "Monash", "Whitehorse"]),
    ["rental_area_name", "service_category", "eligible_departure_events", "unique_trip_records", "served_stop_records"]
])

SERVICE RECONCILIATION PASSED
Source stop-time rows scanned: 11847183
Active-trip stop-time rows: 1311746
LGA/category/window rows: 248
Membership, stop-level and network totals match.


,service_category,window,eligible_departure_events,unique_trip_records,served_stop_records,unique_route_records,extended_time_events
0,train_not_labelled_replacement,service_day,46081,2776,496,16,844
1,labelled_replacement_bus,service_day,480,52,27,3,113
2,train_not_labelled_replacement,am_peak_0700_0900,7347,569,484,16,0
3,labelled_replacement_bus,am_peak_0700_0900,0,0,0,0,0
4,tram,service_day,227716,5044,1607,24,4716
5,tram,am_peak_0700_0900,28642,887,1606,23,0
6,myki_bus,service_day,901004,24178,18825,477,2353
7,myki_bus,am_peak_0700_0900,122943,4312,18358,422,0


,rental_area_name,service_category,eligible_departure_events,unique_trip_records,served_stop_records
68,Melbourne,labelled_replacement_bus,0,0,0
69,Melbourne,myki_bus,4247,499,249
70,Melbourne,train_not_labelled_replacement,1244,437,56
71,Melbourne,tram,8605,697,274
80,Monash,labelled_replacement_bus,0,0,0
81,Monash,myki_bus,7345,503,852
82,Monash,train_not_labelled_replacement,250,82,14
83,Monash,tram,0,0,0
104,Whitehorse,labelled_replacement_bus,0,0,0
105,Whitehorse,myki_bus,6604,431,820


## 13. Save a reproducible result — not a recommendation score

We save nine CSV tables and a manifest, not millions of joined event rows. `event_trace_examples.csv`
is explicitly a sample for tracing filters back to a source line/trip/sequence; all summary calculations
used the complete selected event data.

The files under `data/interim/gtfs_service/` are this notebook's derived outputs. Rerunning this stage
replaces those files and the generated report. It does not modify the spatial assignments, original
archives, rental data, registry, prior reports or application. Staging and per-file replacement are
not an all-or-nothing database transaction: do not run two writers simultaneously.

CSV read-back is checked with identifiers as text, and the LGA numeric counts are checked separately.
Retain unknown/outside records upstream; zero in the final LGA grid means **no counted events under
this date, feed scope and policy**, not proof that residents have no transport.

In [13]:
SERVICE_OUT = ROOT / "data/interim/gtfs_service"
SERVICE_REPORT = ROOT / "docs/gtfs_service_indicators.md"
exports = {
    "active_trip_lookup.csv": active_for_service,
    "trip_stop_scope.csv": trip_stop_scope,
    "lga_service_indicators.csv": lga_service,
    "stop_service_indicators.csv": stop_service,
    "trip_lga_memberships.csv": trip_lga_memberships,
    "feed_service_summary.csv": network_service,
    "event_filter_audit.csv": filter_audit,
    "scan_summary.csv": scan_summary,
    "event_trace_examples.csv": trace_examples,
}
for path, digest in input_hashes.items():
    require(sha256_file(path) == digest, f"Upstream input changed: {path.name}")

totals = {}
for window in WINDOWS:
    view = network_service.loc[network_service.window.eq(window)]
    totals[window] = {
        "eligible_departure_events": int(view.eligible_departure_events.sum()),
        # Categories use disjoint feed/trip keys; network counts already deduplicated across LGAs.
        "unique_trip_records": int(view.unique_trip_records.sum()),
        "extended_time_events": int(view.extended_time_events.sum()),
        "near_boundary_events": int(view.near_boundary_events.sum()),
    }

SERVICE_OUT.mkdir(parents=True, exist_ok=True)
with TemporaryDirectory(prefix="service-stage-", dir=SERVICE_OUT) as temp_dir:
    stage = Path(temp_dir)
    schemas = {}
    for filename, table in exports.items():
        path = stage / filename
        table.to_csv(path, index=False, encoding="utf-8", lineterminator="\n")
        back = pd.read_csv(path, dtype="string", keep_default_na=False)
        expected_text = pd.read_csv(
            io.StringIO(table.to_csv(index=False, lineterminator="\n")),
            dtype="string", keep_default_na=False,
        )
        pd.testing.assert_frame_equal(back, expected_text, check_exact=True)
        schemas[filename] = {"rows": len(table), "columns": table.columns.tolist()}
    lga_back = pd.read_csv(stage / "lga_service_indicators.csv", dtype="string", keep_default_na=False)
    for col in COUNT_FIELDS + ["eligible_events_excluding_near_boundary"]:
        require(np.array_equal(pd.to_numeric(lga_back[col]).to_numpy(dtype="int64"), lga_service[col].to_numpy()),
                f"LGA numeric read-back changed {col}.")
    output_hashes = {name: sha256_file(stage / name) for name in exports}
    manifest = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "notebook": "notebooks/07_gtfs_service_indicators.ipynb",
        "version": "part_a_and_b_v1",
        "source": {
            "gtfs_filename": GTFS_PATH.name, "gtfs_sha256": EXPECTED_GTFS_SHA256,
            "stop_assignments_sha256": input_hashes[STOP_PATH],
            "lga_crosswalk_sha256": input_hashes[LGA_PATH],
            "selected_feeds": list(FEED_IDS),
        },
        "scope": {
            "service_date": SERVICE_DATE.isoformat(), "agency_timezone": "Australia/Melbourne",
            "lga_count": 31, "rental_reporting_quarter": "2025Q3",
            "event_key": EVENT_KEYS, "lga_table_key": LGA_GROUP + ["service_date"],
            "categories": CATEGORY_FEED,
            "windows": {k: {"start_seconds": v[0], "end_seconds_exclusive": v[1]} for k, v in WINDOWS.items()},
            "eligibility_rule": "active trip; pickup 0 including blank; later full-trip stop sequence; matched_within LGA",
            "terminal_rule": "Exclude final stop-time record of each trip; linked-trip/block continuation not modelled",
            "dropoff_rule": "Drop-off codes retained, but downstream alighting/transfer feasibility not solved",
            "after_midnight_rule": "Retain extended times in selected service day; never modulo 24 or merge adjacent service dates",
            "replacement_rule": "Explicit route_short_name Replacement Bus retained as its own category",
            "spatial_boundary_flag": "10 m diagnostic from notebook 06; no reassignment and no accuracy guarantee",
            "not_measured": ["vehicles", "passengers", "route headways", "wait times", "capacity",
                             "walking access", "punctuality", "wheelchair journey feasibility", "suitability score"],
        },
        "quality": {
            "source_stop_time_rows_scanned": int(scan_summary.source_stop_time_rows_scanned.sum()),
            "active_trip_records": len(active_for_service),
            "active_stop_time_rows": int(scan_summary.active_stop_time_rows.sum()),
            "lga_service_rows": len(lga_service),
            "duplicate_active_event_keys": 0,
            "unknown_active_event_trip_or_stop_keys": 0,
            "terminal_exclusions_service_day": int(filter_audit.loc[
                filter_audit.window.eq("service_day"), "terminal_trip_record"].sum()),
            "calendar_self_check_groups": 3,
            "service_policy_self_check_groups": len(service_self_checks),
            "counts_reconciled": True, "csv_readback_matches": True,
            "input_files_unchanged": True, "lga_service_indicators_calculated": True,
        },
        "totals_within_selected_lgas": totals,
        "input_sha256": {p.relative_to(ROOT).as_posix(): h for p,h in input_hashes.items()},
        "output_sha256": output_hashes, "output_schemas": schemas,
        "versions": {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__},
    }
    manifest_name = "service_indicators_manifest.json"
    (stage / manifest_name).write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    require(read_json_object(stage / manifest_name) == manifest, "Manifest read-back changed.")
    for path, digest in input_hashes.items():
        require(sha256_file(path) == digest, f"Upstream input changed during export: {path.name}")
    for name in [*exports, manifest_name]:
        (stage / name).replace(SERVICE_OUT / name)

print("Nine CSV tables saved and reloaded successfully.")
print("Saved:", SERVICE_OUT.relative_to(ROOT))

Nine CSV tables saved and reloaded successfully.
Saved: data\interim\gtfs_service


## 14. Record the findings and final checkpoint

This report is generated from the computed results. Keep personal reflections in a separate Markdown
cell or progress record because rerunning the final cell replaces the report.

The next product step is to review these raw supply indicators before deciding how to compare areas.
A large LGA can have more stops, routes and visits simply because it covers more area. These counts
are not an objective accessibility ranking and are not yet a 0–100 transport score.

In [14]:
report_lines = [
    "# HomeReach Melbourne — Date-specific scheduled service indicators", "",
    f"Generated (UTC): {datetime.now(timezone.utc).isoformat()}", "",
    "## Scope", "",
    f"- Service date: **{SERVICE_DATE.isoformat()}**, Australia/Melbourne GTFS service-day time.",
    "- Rental reporting snapshot remains 2025Q3; do not use this schedule as historical rental-model evidence.",
    "- Operational feeds: 2, 3 and 4; the other five archives are outside the detailed scope.",
    "- Study area: the same 31 reviewed source-group metropolitan LGAs.",
    f"- GTFS source SHA-256: `{EXPECTED_GTFS_SHA256}`.",
    "- Prior stop assignments and crosswalk were verified against their local manifests.", "",
    "## Counting rule", "",
    "One event is one active `(feed_id, trip_id, numeric stop_sequence)` record. Count it only when pickup is regular",
    "(0, including blank), the complete trip has a later stop sequence, and the stop lies inside one selected LGA.",
    "Final records are excluded even if the vehicle might continue as a linked trip. No-pickup (1) and arrangement-required",
    "(2/3) records are kept as separate exclusion counts. This is a scheduled-event proxy; downstream alighting, transfers,",
    "capacity, walking catchments and accessibility are not solved.", "",
    "The complete-trip terminal is calculated BEFORE filtering stops by LGA or departure time.",
    "The full selected service day retains 24:00+ times; the AM window is 07:00 inclusive to 09:00 exclusive.",
    "No modulo-24 conversion is used. Adjacent service-date trips are not merged into a calendar-day inventory.", "",
    "## Counts within the selected LGAs", "",
    "| Category | Window | Counted departure events | Distinct trip records | Served stop records | Distinct route records |",
    "|---|---|---:|---:|---:|---:|",
]
for r in network_service.sort_values(["window", "service_category"]).itertuples(index=False):
    report_lines.append(
        f"| {r.service_category} | {r.window} | {r.eligible_departure_events:,} | "
        f"{r.unique_trip_records:,} | {r.served_stop_records:,} | {r.unique_route_records:,} |"
    )
report_lines += [
    "", "Counts above are distinct across the study area for each category/window. Do not sum LGA trip counts",
    "to obtain a network total: the same trip can provide eligible visits in several LGAs.", "",
    "## Verification", "",
    f"- Source stop-time rows scanned: {int(scan_summary.source_stop_time_rows_scanned.sum()):,}.",
    f"- Active trip records: {len(active_for_service):,}; active stop-time rows: {int(scan_summary.active_stop_time_rows.sum()):,}.",
    f"- Complete LGA/category/window grid: {len(lga_service)} rows (31 × 4 × 2).",
    "- Duplicate active event keys and missing active-event trip/stop references: 0.",
    "- Complete-trip terminal counts, selection partitions, stop totals and trip memberships reconciled.",
    "- Three calendar self-check groups and ten service-policy self-check groups passed.",
    "- Nine CSV exports reloaded and matched; upstream input files remained unchanged.",
    "- No raw timetable rows were overwritten; the registry, previous notebooks and app were not edited.", "",
    "## Reading the output tables", "",
    "- `lga_service_indicators.csv`: the main 248-row analytical table, including zero-count combinations.",
    "- `stop_service_indicators.csv`: counted stop/category/window combinations only; upstream retains all other stops.",
    "- `trip_lga_memberships.csv`: distinct trip/area/window links for independent count verification.",
    "- `active_trip_lookup.csv` and `trip_stop_scope.csv`: date selection and complete-trip scope.",
    "- `feed_service_summary.csv`: counts across the selected LGAs, already deduplicated across areas.",
    "- `event_filter_audit.csv`: exhaustive EXCLUSIVE outcome counts (terminal checked before pickup before location).",
    "- `scan_summary.csv`: source/read/active totals per feed.",
    "- `event_trace_examples.csv`: a labelled sample for checking source line/trip/sequence and exclusion reasons.",
    "- `service_indicators_manifest.json`: source fingerprints, policy, schemas and output fingerprints.", "",
    "## Interpretation cautions", "",
    "These are scheduled stop-record visits, not riders, actual departures, separate vehicles or regular route headways.",
    "A repeated visit on the same trip contributes another event but not another unique trip in the same LGA/window.",
    "A route ID is not necessarily a unique passenger-facing route brand; identical cross-feed IDs remain separate.",
    "The replacement category uses the explicit source label only; non-labelled feed-2 trips remain labelled cautiously.",
    "The Myki feed includes regional services; outside and non-eligible records are excluded from area totals, not erased.",
    "A stored stop outside the polygons can still serve nearby residents. The 10 m boundary flag is diagnostic only.",
    "Zeros mean no counted events under this scope and rule, not proof of no useful transport for residents.",
    "LGA totals are affected by area size and route/station arrangement. No normalisation, score, ranking or forecast is made.", "",
    "## References", "",
    "- https://gtfs.org/documentation/schedule/reference/",
    "- https://opendata.transport.vic.gov.au/dataset/gtfs-schedule",
    "- https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html",
]
SERVICE_REPORT.parent.mkdir(parents=True, exist_ok=True)
SERVICE_REPORT.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
for path, digest in input_hashes.items():
    require(sha256_file(path) == digest, f"Upstream input changed: {path.name}")
print("SERVICE INDICATORS CHECKPOINT PASSED")
print("Service date:", SERVICE_DATE, "(Australia/Melbourne)")
print("Source stop-time rows scanned:", int(scan_summary.source_stop_time_rows_scanned.sum()))
print("Active trip records:", len(active_for_service))
print("Active stop-time rows:", int(scan_summary.active_stop_time_rows.sum()))
print("Counted service-day departure events:", totals["service_day"]["eligible_departure_events"])
print("Counted AM departure events:", totals["am_peak_0700_0900"]["eligible_departure_events"])
print("Distinct AM trip records across the study area:", totals["am_peak_0700_0900"]["unique_trip_records"])
print("LGA/category/window rows:", len(lga_service))
print("Calendar self-check groups: 3 passed")
print("Service-policy self-check groups:", len(service_self_checks), "passed")
print("Independent membership and stop-total reconciliations: passed")
print("Duplicate active event keys and unknown references: 0")
print("CSV tables saved and reloaded:", len(exports))
print("LGA service indicators calculated: True")
print("Upstream inputs unchanged: True")
print("Saved report:", SERVICE_REPORT.relative_to(ROOT))
print("Saved outputs:", SERVICE_OUT.relative_to(ROOT))

SERVICE INDICATORS CHECKPOINT PASSED
Service date: 2026-10-13 (Australia/Melbourne)
Source stop-time rows scanned: 11847183
Active trip records: 35580
Active stop-time rows: 1311746
Counted service-day departure events: 1175281
Counted AM departure events: 158932
Distinct AM trip records across the study area: 5768
LGA/category/window rows: 248
Calendar self-check groups: 3 passed
Service-policy self-check groups: 10 passed
Independent membership and stop-total reconciliations: passed
Duplicate active event keys and unknown references: 0
CSV tables saved and reloaded: 9
LGA service indicators calculated: True
Upstream inputs unchanged: True
Saved report: docs\gtfs_service_indicators.md
Saved outputs: data\interim\gtfs_service


## Checkpoint before continuing

Save this notebook with **Ctrl + S**. Send the final `SERVICE INDICATORS CHECKPOINT PASSED` output
or the first error. No need for screenshots of every cell. You do not need to rerun notebooks 05/06
unless a required earlier output is missing or its verification fails.

You should be able to explain: **we applied service calendars, retained whole-trip sequences before
filtering, excluded terminal/no-pickup/arrangement records under an explicit rule, kept labelled
replacement services separate, and counted stop events and distinct trips differently.**

Part A's intermediate messages remain in this file; only this final checkpoint confirms Part B.
The 125-test pytest suite has not been modified by these notebook self-checks. Neither the live
Streamlit demonstration nor any recommendation score has changed.